<a href="https://colab.research.google.com/github/lcbjrrr/DBMS/blob/main/Lab_ACID.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [50]:
# Download DB
!wget https://raw.githubusercontent.com/lcbjrrr/DBMS/refs/heads/main/ok.sqlite

--2026-08-19 16:51:35--  https://raw.githubusercontent.com/lcbjrrr/DBMS/refs/heads/main/ok.sqlite
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 45056 (44K) [application/octet-stream]
Saving to: ‘ok.sqlite’

ok.sqlite           100%[===================>]  44.00K  --.-KB/s    in 0.001s  

2026-08-19 16:51:35 (53.7 MB/s) - ‘ok.sqlite’ saved [45056/45056]



# Lab Activity: SQL Transactions

A transaction is the mechanism that groups related statements into a single unit of work, so that the database ends up with either all the changes or none, never half. In this activity, you will first create the inconsistency deliberately and see it sitting in the table, then discover that simply wrapping the statements in BEGIN does not rescue you, and finally learn where the real protection comes from: the choice between COMMIT and ROLLBACK after you have seen whether the work actually succeeded.

## Step 1 — Try This Example


Run these two statements one after the other, with no transaction:


In [51]:
import sqlite3
db_path = '/content/ok.sqlite'
conn = None
try:
    conn = sqlite3.connect(db_path,isolation_level=None)
    cursor = conn.cursor()
    cursor.execute("INSERT INTO Publishers (publisher_name, impact_factor) VALUES ('SHOULD NOT INSERT', 3.10);")
    cursor.execute("INSERT INTO Journals (publisher_id, issn, year, volume, number) VALUES (last_insert_rowid(), NULL, 2024, 33, 2);")
    print("SQL statements executed successfully.")
except sqlite3.Error as e:
    print(f"An error occurred: {e}")
finally:
    if conn:
        conn.close()

An error occurred: NOT NULL constraint failed: Journals.issn


In [38]:
!sudo apt install sqlite

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
sqlite is already the newest version (2.8.17-15fakesync1build1).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.


In [52]:
! sqlite3 ok.sqlite "SELECT * FROM Publishers;"

1|Nature Publishing|42.77
2|IEEE|10.5
3|ACM|8.2
4|Springer|5.4
5|Elsevier|6.1
6|NeurIPS Foundation|15
7|O'Reilly Media|2.1
8|SHOULD NOT INSERT|3.1


## Step 2 — Now Talk About Transactions



Same work, but wrapped this time. Write down your prediction before you run it.


In [54]:
import sqlite3
db_path = '/content/ok.sqlite'
conn = None
try:
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("INSERT INTO Publishers (publisher_name, impact_factor) VALUES ('WILL NOT INSERT', 3.10);")
    cursor.execute("INSERT INTO Journals (publisher_id, issn, year, volume, number) VALUES (last_insert_rowid(), NULL, 2024, 33, 2);")
    conn.commit()
    print("SQL statements executed successfully within a transaction.")
except sqlite3.Error as e:
    if conn:
        conn.rollback()
    print(f"An error occurred: {e}. Transaction rolled back.")
finally:
    if conn:
        conn.close()

An error occurred: NOT NULL constraint failed: Journals.issn. Transaction rolled back.


In [55]:
! sqlite3 ok.sqlite "SELECT * FROM Publishers;"

1|Nature Publishing|42.77
2|IEEE|10.5
3|ACM|8.2
4|Springer|5.4
5|Elsevier|6.1
6|NeurIPS Foundation|15
7|O'Reilly Media|2.1
8|SHOULD NOT INSERT|3.1


Now ok!

In [56]:
import sqlite3
db_path = '/content/ok.sqlite'
conn = None
try:
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("INSERT INTO Publishers (publisher_name, impact_factor) VALUES ('NOW IT IS OK', 3.10);")
    cursor.execute("INSERT INTO Journals (publisher_id, issn, year, volume, number) VALUES (last_insert_rowid(), '000', 2024, 33, 2);")
    conn.commit()
    print("SQL statements executed successfully within a transaction.")
except sqlite3.Error as e:
    if conn:
        conn.rollback()
    print(f"An error occurred: {e}. Transaction rolled back.")
finally:
    if conn:
        conn.close()

SQL statements executed successfully within a transaction.


In [57]:
! sqlite3 ok.sqlite "SELECT * FROM Publishers;"

1|Nature Publishing|42.77
2|IEEE|10.5
3|ACM|8.2
4|Springer|5.4
5|Elsevier|6.1
6|NeurIPS Foundation|15
7|O'Reilly Media|2.1
8|SHOULD NOT INSERT|3.1
9|NOW IT IS OK|3.1


## Step 3 - Working with Transactions

Let's read data from `pubs.csv` and `journals.csv`, and inserts it into the `Publishers` and `Journals` tables respectively, all within a single transaction and with error handling.

In [58]:
import sqlite3
import csv
db_path = '/content/ok.sqlite'
pubs_csv_path = '/content/pubs.csv'
journals_csv_path = '/content/journals.csv'
conn = None
try:
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    with open(pubs_csv_path, 'r', newline='', encoding='utf-8') as inst_file:
        inst_reader = csv.reader(inst_file)
        for row in inst_reader:
            if len(row) >= 2:
                id = row[0]
                cursor.execute("INSERT INTO Publishers (publisher_id, publisher_name, impact_factor) VALUES (?, ?, ?);", (id,row[1], row[2]))

    with open(journals_csv_path, 'r', newline='', encoding='utf-8') as authors_file:
        authors_reader = csv.reader(authors_file)
        for row in authors_reader:
            if len(row) >= 2:
                  cursor.execute("INSERT INTO Journals (publisher_id, issn, year, volume) VALUES (?,?,?,?);", (id, row[1], row[2], row[3]))
    conn.commit()
    print("Data from Publishers and Journals imported successfully within a single transaction.")

except Exception as e:
    if conn:
        conn.rollback()
    print(f"An unexpected error occurred: {e}. Transaction rolled back.")
finally:
    if conn:
        conn.close()

An unexpected error occurred: [Errno 2] No such file or directory: '/content/pubs.csv'. Transaction rolled back.


In [59]:
! sqlite3 ok.sqlite "SELECT * FROM Publishers p, Journals j WHERE p.publisher_id = j.publisher_id;"

1|Nature Publishing|42.77|1|1476-4687|2024|625|7995
2|IEEE|10.5|2|0018-9219|2023|111|5
3|ACM|8.2|3|0001-0782|2024|67|2
4|Springer|5.4|4|0028-0836|2022|50|12
5|Elsevier|6.1|5|0022-2836|2023|435|10
9|NOW IT IS OK|3.1|9|000|2024|33|2


### Second run

In [60]:
# Download the CSVs files
!wget https://raw.githubusercontent.com/lcbjrrr/DBMS/refs/heads/main/pubs.csv
!wget https://raw.githubusercontent.com/lcbjrrr/DBMS/refs/heads/main/journals.csv
# And re-run it...

--2026-08-19 16:53:02--  https://raw.githubusercontent.com/lcbjrrr/DBMS/refs/heads/main/pubs.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 19 [text/plain]
Saving to: ‘pubs.csv’

pubs.csv            100%[===================>]      19  --.-KB/s    in 0s      

2026-08-19 16:53:02 (389 KB/s) - ‘pubs.csv’ saved [19/19]

--2026-08-19 16:53:02--  https://raw.githubusercontent.com/lcbjrrr/DBMS/refs/heads/main/journals.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 30 [text/plain]
Saving to: ‘journals.csv’

journals.csv        

In [61]:
import sqlite3
import csv
db_path = '/content/ok.sqlite'
pubs_csv_path = '/content/pubs.csv'
journals_csv_path = '/content/journals.csv'
conn = None
try:
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    with open(pubs_csv_path, 'r', newline='', encoding='utf-8') as inst_file:
        inst_reader = csv.reader(inst_file)
        for row in inst_reader:
            if len(row) >= 2:
                id = row[0]
                cursor.execute("INSERT INTO Publishers (publisher_id, publisher_name, impact_factor) VALUES (?, ?, ?);", (id,row[1], row[2]))

    with open(journals_csv_path, 'r', newline='', encoding='utf-8') as authors_file:
        authors_reader = csv.reader(authors_file)
        for row in authors_reader:
            if len(row) >= 2:
                  cursor.execute("INSERT INTO Journals (publisher_id, issn, year, volume) VALUES (?,?,?,?);", (id, row[1], row[2], row[3]))
    conn.commit()
    print("Data from Publishers and Journals imported successfully within a single transaction.")

except Exception as e:
    if conn:
        conn.rollback()
    print(f"An unexpected error occurred: {e}. Transaction rolled back.")
finally:
    if conn:
        conn.close()

Data from Publishers and Journals imported successfully within a single transaction.


In [62]:
! sqlite3 ok.sqlite "SELECT * FROM Publishers p, Journals j WHERE p.publisher_id = j.publisher_id;"

1|Nature Publishing|42.77|1|1476-4687|2024|625|7995
2|IEEE|10.5|2|0018-9219|2023|111|5
3|ACM|8.2|3|0001-0782|2024|67|2
4|Springer|5.4|4|0028-0836|2022|50|12
5|Elsevier|6.1|5|0022-2836|2023|435|10
9|NOW IT IS OK|3.1|9|000|2024|33|2
99|'PUB 99'|2.1|99| '99-999'|2024|625|
